# RAG from Scratch to Production — Episode 5/20
## Embeddings: search by meaning, not by words

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JakkulaMahendhar/rag-from-scratch/blob/main/notebooks/05_embeddings.ipynb)

Since Episode 1, one bug has followed us: the user asks *"How often can I **WFH**?"*, the handbook says *"**remote**"*, and keyword search finds nothing because they share no words.

I promised embeddings would fix it. In this episode we **test that promise** with the same model the [production app](https://github.com/JakkulaMahendhar/rag-chatbot) uses.

**In this episode:**
1. See what an embedding is: a sentence turned into 384 numbers.
2. Measure similarity yourself: it's just multiplying and adding.
3. Race keyword search against embedding search on 10 questions in everyday language.
4. Ask the WFH question, and see what actually happens.
5. Compare 3 free embedding models.

**No API key needed:** everything here runs locally, in Colab or on your laptop.

## 0. Setup

`sentence-transformers` downloads and runs embedding models. The first model is about 90 MB.

In [ ]:
%pip install -q sentence-transformers

In [1]:
import re
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")   # the production app's model
print("Model loaded")

Model loaded


## 1. What is an embedding?

An embedding model reads a piece of text and returns a list of numbers, a **vector**. The model was trained so that texts with **similar meaning get similar numbers**.

In [2]:
vector = model.encode("Employees may work remotely up to 2 days per week.")

print(f"{len(vector)} numbers")
print(np.round(vector[:8], 3), "...")

384 numbers
[ 0.011 -0.026  0.063 -0.002 -0.042  0.023 -0.068 -0.049] ...


384 numbers for one sentence. On their own they mean nothing to us. What matters is **comparing** two vectors.

## 2. Similarity is just multiply-and-add

To compare two vectors, multiply them number by number and add everything up. This is called the **dot product**. Because this model's vectors all have length 1, the dot product equals the **cosine similarity**: **1** means same meaning, **0** means unrelated.

In [3]:
def similarity(a: str, b: str) -> float:
    va, vb = model.encode([a, b], normalize_embeddings=True)
    return float(np.dot(va, vb))     # multiply number by number, then add


PAIRS = [
    ("work from home",  "remote work"),
    ("vacation",        "annual leave"),
    ("training money",  "learning budget"),
    ("laptop stolen",   "lost or stolen devices"),
    ("work from home",  "learning budget"),
    ("vacation",        "password"),
]
for a, b in PAIRS:
    print(f"{a!r:18} vs {b!r:26} {similarity(a, b):.2f}")

'work from home'   vs 'remote work'              0.54
'vacation'         vs 'annual leave'             0.44
'training money'   vs 'learning budget'          0.49
'laptop stolen'    vs 'lost or stolen devices'   0.63
'work from home'   vs 'learning budget'          0.06
'vacation'         vs 'password'                 0.21


**"Vacation"** and **"annual leave"** share zero words, yet score far higher than **"vacation"** and **"password"**. That is exactly what keyword search could never do.

## 3. Keyword search vs embedding search

The same 6-section handbook as Episode 4, one chunk per section (what Episode 4's recursive splitter produced at 300 characters):

In [4]:
HANDBOOK = """NovaTech Solutions - Employee Handbook (2026 edition)

Remote Work
Employees may work remotely up to 2 days per week. Wednesday is a mandatory in-office day for all teams. Remote days must be logged in the HR portal by Friday of the previous week. Managers may approve extra remote days during school holidays.

Annual Leave
Full-time employees receive 24 days of paid annual leave per calendar year. A maximum of 5 unused days may be carried over into the next year; the rest expire on 31 March. Leave requests of more than 10 consecutive days need approval at least one month in advance.

Expenses
Business expenses must be submitted within 30 days with a receipt. The meal allowance during business travel is capped at 40 USD per day. Expense claims above 500 USD must also be approved by the finance director before payment.

Learning Budget
Every employee has an annual learning budget of 1,200 USD for courses, certifications and conferences. It does not roll over: whatever is not spent by 31 December is lost. Requests are approved by your direct manager.

Equipment
Every employee receives a laptop, refreshed every 3 years. Lost or stolen devices must be reported to IT within 24 hours. Personal devices may only access company email after enrolling in device management.

Security
Multi-factor authentication is mandatory on all company accounts. Suspected security incidents must be reported to the security team within 1 hour. Passwords must be at least 14 characters long and must never be reused."""

chunks = HANDBOOK.split("\n\n")[1:]
print([c.split("\n")[0] for c in chunks])

['Remote Work', 'Annual Leave', 'Expenses', 'Learning Budget', 'Equipment', 'Security']


**Embedding search** works in two steps, just like production:
1. **Once, at upload:** embed every chunk and store the vectors (production stores them in ChromaDB, Episode 6).
2. **Per question:** embed the question, score it against every chunk vector, return the best.

In [5]:
chunk_vectors = model.encode(chunks, normalize_embeddings=True)    # step 1: once
print(chunk_vectors.shape, "→ 6 chunks × 384 numbers")


def search_embeddings(question: str) -> tuple[str, float]:
    q = model.encode(question, normalize_embeddings=True)          # step 2: per question
    scores = chunk_vectors @ q             # one dot product per chunk, all at once
    best = int(np.argmax(scores))
    return chunks[best], float(scores[best])

(6, 384) → 6 chunks × 384 numbers


And Episode 1's **keyword search**, for comparison. It returns nothing when no words match:

In [6]:
STOPWORDS = {"a", "an", "the", "is", "are", "and", "or", "of", "to", "in", "on",
             "at", "for", "per", "what", "how", "many", "much", "does", "do", "i",
             "my", "can", "get", "it", "be", "must", "by", "with", "who", "when"}

def words(text: str) -> set[str]:
    return {w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOPWORDS}


def search_keywords(question: str) -> tuple[str, float]:
    q = words(question)
    scores = [len(q & words(c)) / len(q) for c in chunks]
    best = int(np.argmax(scores))
    return (chunks[best], scores[best]) if scores[best] > 0 else ("", 0.0)

Now the race. 10 questions written the way employees actually talk, each with a fact the retrieved chunk **must** contain (Episode 4's known-facts test):

In [7]:
EVERYDAY_QUESTIONS = [
    ("Can I save unused training money for next year?",       "not roll over"),
    ("My laptop got stolen, who do I tell?",                   "24 hours"),
    ("Can I carry vacation days into next year?",              "5 unused days"),
    ("Who signs off on big expense claims?",                   "finance director"),
    ("How long should my password be?",                        "14 characters"),
    ("Do I need 2FA on my work accounts?",                     "Multi-factor"),
    ("What do I get for food when travelling for work?",       "40 USD"),
    ("Is Wednesday a home office day?",                        "Wednesday"),
    ("I clicked a phishing link, how fast must I report it?",  "1 hour"),
    ("How often can I work from home?",                        "2 days per week"),
]

def title(chunk: str) -> str:
    return chunk.split("\n")[0] if chunk else "(nothing)"

totals = {"keywords": 0, "embeddings": 0}
print(f"{'question':<52} {'keywords':<16} {'embeddings'}")
for question, fact in EVERYDAY_QUESTIONS:
    row = []
    for name, search in [("keywords", search_keywords), ("embeddings", search_embeddings)]:
        chunk, _ = search(question)
        hit = fact in chunk
        totals[name] += hit
        row.append(f"{'✅' if hit else '❌'} {title(chunk):<13}")
    print(f"{question[:50]:<52} {row[0]} {row[1]}")
print(f"\n{'TOTAL':<52} keywords {totals['keywords']}/10    embeddings {totals['embeddings']}/10")

question                                             keywords         embeddings
Can I save unused training money for next year?      ❌ Annual Leave  ✅ Learning Budget


My laptop got stolen, who do I tell?                 ✅ Equipment     ✅ Equipment    
Can I carry vacation days into next year?            ✅ Annual Leave  ✅ Annual Leave 
Who signs off on big expense claims?                 ✅ Expenses      ✅ Expenses     
How long should my password be?                      ✅ Security      ✅ Security     
Do I need 2FA on my work accounts?                   ❌ Remote Work   ✅ Security     
What do I get for food when travelling for work?     ❌ Remote Work   ✅ Expenses     
Is Wednesday a home office day?                      ✅ Remote Work   ✅ Remote Work  
I clicked a phishing link, how fast must I report    ❌ (nothing)     ❌ Equipment    
How often can I work from home?                      ✅ Remote Work   ✅ Remote Work  

TOTAL                                                keywords 6/10    embeddings 9/10


Embeddings win clearly. They connect **"training money"** to *learning budget*, **"2FA"** to *multi-factor authentication*, and **"food when travelling"** to *meal allowance*, without sharing a single word.

(Not every question is solved. More on the phishing one in section 5.)

## 4. The promise: does "WFH" work now?

In [8]:
for question in ["How often can I work from home?", "How often can I WFH?"]:
    q = model.encode(question, normalize_embeddings=True)
    scores = chunk_vectors @ q
    print(f"Q: {question}")
    for i in np.argsort(-scores)[:3]:
        print(f"   {scores[i]:.2f}  {title(chunks[i])}")
    print()

print(f"'WFH' vs 'remote work':     {similarity('WFH', 'remote work'):.2f}")
print(f"'WFH' vs 'learning budget': {similarity('WFH', 'learning budget'):.2f}")

Q: How often can I work from home?
   0.48  Remote Work
   0.38  Annual Leave
   0.32  Equipment

Q: How often can I WFH?
   0.32  Annual Leave
   0.28  Remote Work
   0.21  Equipment

'WFH' vs 'remote work':     0.18
'WFH' vs 'learning budget': 0.08


**No, it doesn't.** I promised embeddings would fix this, so here's the honest result:

- *"work from home"* → **Remote Work**, clearly. The model understands the meaning.
- *"WFH"* → **Annual Leave**, with Remote Work a close second. The production model barely knows the abbreviation: "WFH" vs "remote work" scores just **0.18**, compared with **0.54** for "work from home" vs "remote work".

Embeddings only know what their training data taught them. A small, fast model like `all-MiniLM-L6-v2` (90 MB) has seen "work from home" millions of times, and "WFH" far less. Abbreviations, internal jargon, product names and codes are exactly where small embedding models struggle.

## 5. Would a different model help?

Let's try two other free models on the hard questions. This downloads about 550 MB more.

In [9]:
MODELS = {
    "all-MiniLM-L6-v2 (production)": "all-MiniLM-L6-v2",
    "bge-small-en-v1.5":             "BAAI/bge-small-en-v1.5",
    "all-mpnet-base-v2":             "all-mpnet-base-v2",
}
HARD = [
    ("How often can I WFH?",                                  "2 days per week"),
    ("I clicked a phishing link, how fast must I report it?", "1 hour"),
]

for label, name in MODELS.items():
    m = SentenceTransformer(name)
    vectors = m.encode(chunks, normalize_embeddings=True)
    wfh, remote, budget = m.encode(["WFH", "remote work", "learning budget"], normalize_embeddings=True)
    print(f"── {label}  ({vectors.shape[1]} numbers per text)")
    print(f"   'WFH' vs 'remote work' {wfh @ remote:.2f}   'WFH' vs 'learning budget' {wfh @ budget:.2f}")
    for question, fact in HARD:
        scores = vectors @ m.encode(question, normalize_embeddings=True)
        best = chunks[int(np.argmax(scores))]
        print(f"   {'✅' if fact in best else '❌'} {question[:48]:<50} → {title(best)}")

── all-MiniLM-L6-v2 (production)  (384 numbers per text)
   'WFH' vs 'remote work' 0.18   'WFH' vs 'learning budget' 0.08
   ❌ How often can I WFH?                               → Annual Leave
   ❌ I clicked a phishing link, how fast must I repor   → Equipment


── bge-small-en-v1.5  (384 numbers per text)
   'WFH' vs 'remote work' 0.50   'WFH' vs 'learning budget' 0.56
   ✅ How often can I WFH?                               → Remote Work
   ✅ I clicked a phishing link, how fast must I repor   → Security


── all-mpnet-base-v2  (768 numbers per text)
   'WFH' vs 'remote work' 0.12   'WFH' vs 'learning budget' -0.01
   ✅ How often can I WFH?                               → Remote Work
   ❌ I clicked a phishing link, how fast must I repor   → Equipment


- **bge-small-en-v1.5** (about the same size as production's model) answers **both** hard questions.
- **all-mpnet-base-v2** (5× bigger) fixes WFH but still misses phishing.
- Look at the raw scores: bge-small rates "WFH" closer to "learning budget" than to "remote work" as isolated words, yet it **still retrieves the right section** for the full question. Each model uses its own scale. A similarity threshold tuned for one model (production uses one) means nothing for another.

So "use embeddings" isn't the end of the story. **Which** embedding model matters, and the only way to choose is to test on your own questions.

## 6. What production does

The [production app](https://github.com/JakkulaMahendhar/rag-chatbot) embeds every chunk with `all-MiniLM-L6-v2` in the background worker (`app/services/embedding.py`) and stores the vectors in ChromaDB. ChromaDB measures **distance** instead of similarity, and the app converts it to a score with `1 / (1 + distance)`. For vectors of length 1, ranking by distance and ranking by cosine similarity give the same order.

The app also doesn't rely on embeddings alone:
- **Query expansion** (Episode 2) rewrites "WFH" into phrasings like "work from home", which this model handles well.
- **Hybrid search** (Episodes 7–8) adds keyword matching back for exact terms like codes and names.

## Recap

| | Keyword search | Embedding search |
|---|---|---|
| Matches | Exact words | Meaning |
| "training money" → learning budget | ❌ | ✅ |
| "work from home" → remote work | ❌ | ✅ |
| "WFH" → remote work | ❌ | ❌ with production's model, ✅ with some others |
| Needs | Nothing | An embedding model, run once per chunk |

**The rule:** embeddings fix most wording mismatches, not all. Abbreviations and jargon are their weak spot. **Test the model on your own questions** before trusting it.

**Next, Episode 6:** we have vectors. Now we store and search millions of them, fast, with a **vector database** (ChromaDB).

⭐ Notebooks: https://github.com/JakkulaMahendhar/rag-from-scratch
🏗️ Full production app: https://github.com/JakkulaMahendhar/rag-chatbot
📸 Instagram: [@codingwithmahi](https://instagram.com/codingwithmahi)